# 04 - 最小可跑的 Pretrain

> 配套笔记：[../notes/03_预训练 Pretrain.md](../notes/03_预训练%20Pretrain.md)

把 doc 里整条 pipeline 跑通的最小版本——wikitext-2 + tiny Qwen2 配置。**不是真训能用的 base model**（数据量 / 模型规模都远不够），只把 pipeline 走通。

**运行环境与耗时**：

| 环境                    | 预计总耗时（含下载）| 备注                                            |
| ----------------------- | ------------------ | ----------------------------------------------- |
| **Colab T4（作者实测）** | **~2 min**         | 训练本身 ~47s，其余是 tokenizer / wikitext 下载 |
| Colab CPU               | > 30 min            | 同 pipeline，纯 fp32                            |

**依赖**：`transformers / datasets / accelerate`，首次执行 §0 安装；不需要 GPU、不需要 HF_TOKEN（wikitext + Qwen tokenizer 都是 public）。

**对应 doc 的章节**：

| Notebook §          | Doc §                                                |
| ------------------- | ---------------------------------------------------- |
| 1. 数据             | §2 数据 + §3.1 input 形状 + §3.2 tokenize + §3.3 chunk |
| 2. 模型 + 训练      | §3.4 labels + §4.1 loss/grad + §4.2 lr 调度          |
| 3. 评估             | §4.3 评估                                            |
| 4. Smoke test       | §5 base model 能做什么                                |

doc 里 §1（pretrain 位置）和 §2 里数据来源 / dedup 这些是纯概念部分，notebook 直接跳过。

## 0. 环境

In [1]:
!nvidia-smi

Sat May  2 06:48:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets accelerate

In [3]:
import torch
from itertools import chain
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, Qwen2Config,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)

## 1. 数据

> 对应 doc §2 数据 + §3.1~3.3 input → tokenize → chunk

三步走：**加载 raw → tokenize 每篇文档 → concat 成长流再切定长 chunk**。

doc §2 讲到工业级 pretrain 数据（CommonCrawl ~50TB → RefinedWeb ~5TB）+ 复杂的 filter / dedup pipeline，这里直接跳过用清洗好的 wikitext-2（~2M tokens），等同于 doc §2.3 prepared 后的数据。

In [4]:
# Tokenizer：复用 Qwen2.5 的（vocab=151,936），免去自己训词表
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
EOS = tokenizer.eos_token  # "<|im_end|>"
tokenizer.pad_token = tokenizer.eos_token

# 用 len(tokenizer) 拿真实 vocab size（含 added tokens），避免 token id 越过 embedding 行数
VOCAB_SIZE = len(tokenizer)
print(f"VOCAB_SIZE = {VOCAB_SIZE}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

VOCAB_SIZE = 151665


In [5]:
# 加载 wikitext-2（≈ doc §2.3 prepared 后的数据），filter 掉空行
raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
raw = raw.filter(lambda x: len(x["text"].strip()) > 0)
print(f"原始样本数: {len(raw)}")
print(f"第 0 条文档: {raw[0]['text'][:80]!r}")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

原始样本数: 23767
第 0 条文档: ' = Valkyria Chronicles III = \n'


In [6]:
# Tokenize + Concat + Chunk（= doc §3.2 + §3.3）
# 每篇尾部加 <eos> → 拼成一条长流 → 按 BLOCK_SIZE 切等长块
BLOCK_SIZE = 256

def tokenize_and_chunk(examples):
    texts = [t + EOS for t in examples["text"]]
    tok = tokenizer(texts, add_special_tokens=False)
    flat = {k: list(chain(*v)) for k, v in tok.items()}
    total = (len(flat["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    return {
        k: [v[i:i+BLOCK_SIZE] for i in range(0, total, BLOCK_SIZE)]
        for k, v in flat.items()
    }

dataset = raw.map(
    tokenize_and_chunk,
    batched=True,
    batch_size=1000,
    remove_columns=raw.column_names,
)
print(f"chunk 数: {len(dataset)}, 每条形状: [{len(dataset[0]['input_ids'])}]")

Map:   0%|          | 0/23767 [00:00<?, ? examples/s]

chunk 数: 9913, 每条形状: [256]


In [7]:
# 看一条 chunk（印证 doc §3.3.1：几篇文档拼到一起 + <|endoftext|> 分隔）
sample = dataset[0]["input_ids"]
print(f"shape: [{len(sample)}]")
print(f"前 30 个 token id: {sample[:30]}")
print(f"反 tokenize 前 80 个 token:\n{tokenizer.decode(sample[:80])}")

shape: [256]
前 30 个 token id: [284, 85162, 88, 4204, 65316, 14429, 284, 715, 151643, 5363, 73, 55661, 902, 85162, 88, 4204, 220, 18, 549, 1230, 8548, 291, 65316, 320, 10769, 549, 49434, 99, 74167, 15767]
反 tokenize 前 80 个 token:
 = Valkyria Chronicles III = 
<|endoftext|> Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation


## 2. 模型 + 训练

> 对应 doc §3.4 labels + §4.1 logits→loss→grad + §4.2 lr 调度

三块拼起来：

- **模型**：用 `Qwen2Config` 拼一个 tiny 配置（≈ 40M params，含 embedding），从头随机初始化
- **Collator** = doc §3.4：`DataCollatorForLanguageModeling(mlm=False)` 做的就是 `labels = input_ids.clone()`，shift by 1 在模型 forward 内部做
- **TrainingArguments** = doc §4.2：`warmup + cosine` 调度；为了几分钟看到 loss 下降，lr 调到 1e-3（比工业级 1e-4 大 10x）

In [8]:
# Tiny Qwen2 配置——CPU 也能训
config = Qwen2Config(
    hidden_size=128,
    intermediate_size=512,
    num_hidden_layers=4,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=VOCAB_SIZE,
    max_position_embeddings=512,
)
model = AutoModelForCausalLM.from_config(config)
print(f"参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

参数量: 39.81 M


In [9]:
# Collator (doc §3.4)：labels = input_ids.clone()
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# TrainingArguments (doc §4.2)：warmup + cosine
args = TrainingArguments(
    output_dir="./pt_out",
    learning_rate=1e-3,                # demo：放大 10x，几百步就能看到 loss 降
    warmup_steps=15,
    lr_scheduler_type="cosine",
    max_steps=300,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    logging_steps=20,
    save_strategy="no",                # demo 不存 checkpoint
    report_to="none",                  # 不连 wandb
    bf16=False,                        # 标准 Colab 上稳定起见
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=collator,
)
trainer.train()

Step,Training Loss
20,11.299829
40,8.778352
60,7.430560
80,7.068898
100,6.803530
120,6.691441
140,6.556960
160,6.404268
180,6.423611
200,6.460188


TrainOutput(global_step=300, training_loss=7.0241526540120445, metrics={'train_runtime': 39.3445, 'train_samples_per_second': 61.0, 'train_steps_per_second': 7.625, 'total_flos': 75196425830400.0, 'train_loss': 7.0241526540120445, 'epoch': 0.24193548387096775})

## 3. 评估：loss 曲线

> 对应 doc §4.3 评估

demo 时长太短没法跑下游 benchmark，只看最基本的 training loss 曲线。预期：起点 ≈ $\ln(\text{VOCAB\_SIZE}) \approx 11.9$，几百步内降到 6~7（说明 pipeline 通了，gradient 在更新参数）。

In [10]:
import pandas as pd
history = pd.DataFrame(trainer.state.log_history)
train_logs = history[history["loss"].notna()][["step", "loss", "learning_rate"]]
print(train_logs.to_string(index=False))

 step      loss  learning_rate
   20 11.299829   9.995140e-04
   40  8.778352   9.826045e-04
   60  7.430560   9.423333e-04
   80  7.068898   8.806501e-04
  100  6.803530   8.005406e-04
  120  6.691441   7.058828e-04
  140  6.556960   6.012587e-04
  160  6.404268   4.917330e-04
  180  6.423611   3.826075e-04
  200  6.460188   2.791647e-04
  220  6.331019   1.864118e-04
  240  6.348736   1.088389e-04
  260  6.277918   5.020105e-05
  280  6.274464   1.333670e-05
  300  6.212515   3.037705e-08


## 4. Smoke Test：base model 续写

> 对应 doc §5 训完的 base model

doc §5 的核心论点：base model 没有对话能力，只能 **continue the text（续写）**。这里给一段 prompt 让模型自回归生成，验证 forward + generate 通了。

因为模型太小 + 训得太短，输出大概率是乱码——能跑通而不报错就算 pipeline 验证成功。

In [11]:
model.eval()
device = next(model.parameters()).device

prompt = "The quick brown fox"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

with torch.no_grad():
    out = model.generate(
        input_ids,
        max_new_tokens=30,
        do_sample=True,
        top_p=0.9,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

print("prompt:", prompt)
print("续写:  ", tokenizer.decode(out[0], skip_special_tokens=True))

prompt: The quick brown fox
续写:   The quick brown fox in the first @-@ episode of the time of 1102 and 1823 the , an 2 @-@


## 离工业级 pretrain 的差距

上面这版只为跑通流程，离能用差太远：

| 维度          | 这版 demo             | 工业级 pretrain                          |
| ------------- | --------------------- | ---------------------------------------- |
| 模型规模      | ~40M params           | 7B ~ 70B+                                |
| 数据量        | ~2M tokens (wikitext) | 1T ~ 15T tokens（混合多源）              |
| BLOCK_SIZE    | 256                   | 4096 ~ 32768                             |
| 训练步数      | 300 steps             | 几十万 ~ 几百万 steps                    |
| 硬件          | 单卡（CPU 也能跑）    | 数千卡 H100/A100，几周到几个月           |
| Document mask | 没开                  | 开（FlashAttention varlen + cu_seqlens） |
| 配比 / 清洗   | 单一来源              | 数据团队多月调比例 + dedup + 质检        |

本质上和上面这个 demo 是同一套流程，只是规模放大。